In [ ]:
!pip install shap transformers librosa datasets

In [ ]:
import torch
import numpy as np
import shap
import matplotlib.pyplot as plt

from datasets import load_dataset, Audio, DatasetDict
from transformers import AutoModelForAudioClassification, AutoFeatureExtractor

import os
from scipy.interpolate import interp1d

### Δείγμα Αναφοράς

In [ ]:
TARGET_IDX = 35

### Φόρτωση finetuned μοντέλου

In [ ]:
repo_id = "VasilisAsim/data2vec-finetuned-TESS_full"
model = AutoModelForAudioClassification.from_pretrained(repo_id)
feature_extractor = AutoFeatureExtractor.from_pretrained(repo_id)

model.eval()

### Φόρτωση dataset (speaker-independent)

In [ ]:
from datasets import load_dataset, Audio, DatasetDict

def load_tess_speaker_independent():
    dataset = load_dataset("AbstractTTS/TESS", split="train")
    dataset = dataset.cast_column("audio", Audio(decode=False))   # για το filter

    def get_speaker(example):
        filename = example["audio"]["path"]
        if "OAF" in filename:
            return "OAF"
        elif "YAF" in filename:
            return "YAF"
        return "unknown"

    train_dataset = dataset.filter(lambda x: get_speaker(x) == "OAF")
    test_dataset = dataset.filter(lambda x: get_speaker(x) == "YAF")

    # ⚠️ ΑΥΤΟ ΤΟ ΒΗΜΑ είναι πιθανότατα αυτό που λείπει:
    train_dataset = train_dataset.cast_column("audio", Audio(sampling_rate=16_000))
    test_dataset = test_dataset.cast_column("audio", Audio(sampling_rate=16_000))

    return DatasetDict({"train": train_dataset, "test": test_dataset})

In [ ]:
dataset = load_tess_speaker_independent()
test_dataset = dataset["test"]

###Συνάρτηση κανονικοποίησης attribution

In [ ]:
def normalise_attribution(attr):
   """
  Κανονικοποίηση attribution που είναι ήδη σε μορφή n_segments τιμών
  (KernelSHAP, DeepSHAP) — η resample_attribution αντίθετα, που χρησιμοποιείται
  σε IG/IxG/CAM/GradientSHAP, χρειάζεται interpolation αφού η μέθοδος παράγει ήδη
  τιμή ανά segment, όχι ανά δείγμα του raw σήματος.
  """
  attr = np.abs(attr)
  return attr / (attr.sum() + 1e-10)

save_dir = "/content/drive/MyDrive/Colab Notebooks/SER_Thesis/01_data2vec_TESS/attributions"
os.makedirs(save_dir, exist_ok=True)

###Επιλογή δείγματος και πρόβλεψης

In [ ]:
sample_index = TARGET_IDX

sample = test_dataset[sample_index]
audio = sample["audio"]["array"]
sampling_rate = sample["audio"]["sampling_rate"]
true_label = sample["emotion"]

In [ ]:
inputs = feature_extractor(audio, sampling_rate=sampling_rate, return_tensors="pt")

with torch.no_grad():
    outputs = model(**inputs)

probabilities = torch.softmax(outputs.logits, dim=-1)

predicted_class = probabilities.argmax(-1).item()

print("True label:",
      true_label)

print("Predicted label:",
      model.config.id2label[predicted_class])

In [ ]:
n_segments = 50

segments = np.array_split(audio, n_segments)

###Συνάρτηση segment-based masking

In [ ]:
def reconstruct_audio(mask):
    """
    Ανακατασκευάζει το audio σήμα από μια binary μάσκα μήκους n_segments:
    segments με mask[i]==1 διατηρούνται, ενώ segments με mask[i]==0 αντικαθίστανται
    με μηδενικά. Χρησιμοποιείται από την predict() για να δημιουργήσει τις
    εκδοχές του σήματος που χρειάζεται το KernelSHAP.
    """
    reconstructed = []

    for i in range(n_segments):
        if mask[i] == 1:
            reconstructed.extend(segments[i])
        else:
            reconstructed.extend(
                np.zeros_like(segments[i])
            )

    return np.array(reconstructed)

In [ ]:
def predict(mask_batch):
    """
    Wrapper για το μοντέλο, στη μορφή που περιμένει το SHAP
    KernelExplainer: παίρνει ένα batch από binary masked segments,
    ανακατασκευάζει το αντίστοιχο audio κάθε μάσκας, και επιστρέφει τις
    πιθανότητες του μοντέλου για κάθε δείγμα.
    """
    outputs = []

    for mask in mask_batch:
        audio_input = reconstruct_audio(mask)
        inputs = feature_extractor(audio_input, sampling_rate=16000, return_tensors="pt")

        with torch.no_grad():
            logits = model(**inputs).logits

        probs = torch.softmax(logits,dim=-1)
        outputs.append(probs.squeeze().numpy())

    return np.array(outputs)

###Υπολογισμός KernelSHAP

In [ ]:
# Baseline: όλα τα segments μηδενισμένα
background = np.zeros((1, n_segments))

In [ ]:
explainer = shap.KernelExplainer(predict, background)

In [ ]:
sample_mask = np.ones((1, n_segments))

# με nsamples η σταθερότητα του KernelSHAP βελτιώνεται
shap_values = explainer.shap_values(sample_mask, nsamples=750)

### Οπτικοποίηση αποτελεσμάτων

In [ ]:
plt.figure(figsize=(12,4))

plt.bar(
    np.arange(n_segments),
    shap_values[0, :, predicted_class]
)

plt.xlabel("Audio Segment")

plt.ylabel("SHAP Value")

plt.title(
    f"SHAP explanation for "
    f"{model.config.id2label[predicted_class]}"
)

plt.show()

### Αποθήκευση attribution

In [ ]:
# (shap_values shape: (1, n_segments, n_classes) — αποθήκευση προβλεπόμενης κλάσης)
kernel_attr = shap_values[0, :, predicted_class]

normalized_kernel = normalise_attribution(kernel_attr)

np.save(f"{save_dir}/KernelSHAP_idx{TARGET_IDX}.npy", normalized_kernel)
print(f"Αποθηκεύτηκε: KernelSHAP_idx{TARGET_IDX}.npy, shape={normalized_kernel.shape}")